# Notebook 3 — Text-to-Image Search with Git-RSCLIP

### *Hands-On Session: Earth Embeddings for EO — Retrieval, Discovery, and Change-Oriented Search*

**Instructors:** Fuxun Yu, Rishi Madhok (TerraByte AI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iceysteel/ESA-NASA-Workshop-2026/blob/main/Day%202/Track%202/Earth-Embeddings-EO/notebook_3_text_to_image_search.ipynb)

---

## What we're doing

Notebooks 0–2 worked in a *single* representation space: classify, find similar patches, etc. Now we cross modalities: **type an English description → retrieve the matching satellite image**.

The model: [`lcybuaa/Git-RSCLIP-base`](https://huggingface.co/lcybuaa/Git-RSCLIP-base) — a SigLIP-style CLIP variant trained on *Git10M*, a 10-million remote-sensing image-text pair corpus. Image and text encoders project into a **shared 768-dim embedding space** where cosine similarity ≈ semantic match.

The data: **EuroSAT test split, 5,400 images, 10 land-cover classes**. To stay inside the 5-minute CPU budget we **don't** re-encode the image pool at workshop time — that would take 30 min on Colab CPU. Instead, the embeddings were precomputed once with the same Git-RSCLIP model (see `scripts/build_eurosat_gitrsclip_embeddings.py`) and shipped as a 7.7 MB `.npz`. At workshop time the notebook only does the cheap part: **encode one text query** (~300 ms CPU) and rank the 5,400 pre-computed image vectors.

## Pipeline

1. Download precomputed image embeddings (`.npz`).
2. Download a small slice of the EuroSAT test images for display (~30 MB).
3. Load Git-RSCLIP **text tower only** for query encoding.
4. Encode any English query → cosine-similarity ranking → display top-K satellite images.


## Runtime

**CPU is fine** — this notebook is designed to fit a 5-minute budget on Colab CPU. The expensive image-pool encoding happens *offline*. T4 GPU shaves the model-load time but isn't required.


In [ ]:
!pip install -q transformers sentencepiece datasets huggingface_hub


In [ ]:
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModel, AutoProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


## 1. Download the precomputed image embedding pool

5,400 image vectors × 768 dims (float16, L2-normalized) were computed **once, offline** using the same Git-RSCLIP image tower we'll use for the text side. Re-encoding 5,400 images at workshop time would cost ~20 min of Colab CPU — embedding once and shipping the result is the only way this fits a 5-minute budget.

The pool lives on the Hugging Face Hub at [`zterrabyte/eurosat-gitrsclip-embeddings`](https://huggingface.co/datasets/zterrabyte/eurosat-gitrsclip-embeddings). `hf_hub_download` pulls it (cached) to the Colab working directory — a few seconds the first time, instant on re-run. (See `scripts/build_eurosat_gitrsclip_embeddings.py` for how the pool was built.)


In [ ]:
HF_POOL_REPO = 'zterrabyte/eurosat-gitrsclip-embeddings'
HF_POOL_FILE = 'eurosat_test_gitrsclip.npz'

t0 = time.time()
npz_path = hf_hub_download(repo_id=HF_POOL_REPO, filename=HF_POOL_FILE, repo_type='dataset')
print(f'Downloaded in {time.time()-t0:.1f}s — {Path(npz_path).stat().st_size/1e6:.1f} MB')

blob = np.load(npz_path, allow_pickle=True)
img_embeddings = blob['embeddings'].astype(np.float32)
labels = blob['labels']
class_names = [str(c) for c in blob['class_names']]
print(f'Pool: {img_embeddings.shape} | classes: {class_names}')


## 2. Load matching EuroSAT images for display

Embeddings let us *rank*; we need the actual images to *show* the participants. We pull the EuroSAT test split (5,400 RGB chips, 64×64, parquet-backed, ~30 MB) — the same dataset used to build the embeddings. The ordering matches row-for-row by construction.


In [ ]:
t0 = time.time()
ds = load_dataset('timm/eurosat-rgb', split='test')
print(f'Loaded {len(ds)} images in {time.time()-t0:.1f}s')
assert len(ds) == len(img_embeddings), 'pool size mismatch'


### Visual check — mixed EuroSAT retrieval pool

Before searching, display a mixed random sample from the EuroSAT pool. Retrieval is not class-conditioned, so this view better matches what the model actually ranks: one shared pool of 5,400 images. Class labels are shown only as lightweight context.


In [ ]:
def show_eurosat_mixed_grid(rows: int = 4, cols: int = 8, seed: int = 7):
    rng = np.random.default_rng(seed)
    n = rows * cols
    chosen = rng.choice(len(ds), size=n, replace=False)
    fig, axes = plt.subplots(rows, cols, figsize=(1.65 * cols, 1.9 * rows), squeeze=False)

    for ax, idx in zip(axes.ravel(), chosen):
        ax.imshow(ds[int(idx)]['image'])
        # ax.set_title(class_names[labels[idx]], fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])

    plt.suptitle('EuroSAT display pool — mixed random examples', y=1.01, fontsize=12)
    plt.tight_layout()
    plt.show()


show_eurosat_mixed_grid(rows=4, cols=8)


## 3. Load the Git-RSCLIP text tower

We only need the **text** side now (the image side already ran offline). Loading the full model is simplest; the text tower is small and we'll only feed it one query at a time.

*The first run downloads ~870 MB of weights — give it 30–60 s on a fresh Colab.*


In [ ]:
MODEL_ID = 'lcybuaa/Git-RSCLIP-base'
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).eval().to(device)
print(f'Model loaded in {time.time()-t0:.1f}s '
      f'({sum(p.numel() for p in model.parameters())/1e6:.0f}M params, '
      f'embed_dim={model.config.text_config.hidden_size})')


## 4. The text-to-image search function

Three steps:

1. Tokenize and run the **text tower** → pooler output → L2-normalize. This is the cheap (~300 ms CPU) part of the whole pipeline.
2. Cosine similarity = `pool_embeddings @ text_vec` (a single 5400×768 mat-vec — instant).
3. Pick top-K, return their image objects and class labels.


In [ ]:
def encode_text(query: str) -> np.ndarray:
    inputs = processor(text=[query], padding='max_length', return_tensors='pt').to(device)
    with torch.inference_mode():
        out = model.text_model(**inputs)
    vec = F.normalize(out.pooler_output.float(), dim=-1).cpu().numpy()[0]
    return vec

def search(query: str, k: int = 8):
    t0 = time.time()
    qvec = encode_text(query)
    enc_ms = (time.time() - t0) * 1000
    sims = img_embeddings @ qvec
    top = np.argsort(-sims)[:k]
    return top, sims[top], enc_ms

def show_results(query: str, k: int = 8):
    top, scores, ms = search(query, k)
    fig, axes = plt.subplots(1, k, figsize=(2 * k, 2.6))
    for ax, idx, score in zip(axes, top, scores):
        ax.imshow(ds[int(idx)]['image']); ax.axis('off')
        # ax.set_title(f'{class_names[labels[idx]]}\ncos={score:.3f}', fontsize=8)
    plt.suptitle(f'Query: "{query}"   (text encode: {ms:.0f} ms)', y=1.05, fontsize=11)
    plt.tight_layout(); plt.show()


### Single-class queries — does it pick the right land cover?

In [ ]:
show_results('a satellite image of a dense forest', k=8)
show_results('a winding river', k=8)
show_results('a dense urban residential neighbourhood', k=8)
show_results('a highway cutting through farmland', k=8)


### Going beyond labelled classes

EuroSAT only has 10 labels. The point of CLIP is that the embedding space is *open vocabulary* — we can describe things the dataset never labelled and still get something sensible back.


In [ ]:
show_results('blue ocean water with no land', k=8)
show_results('parallel rows of crops on a field', k=8)
show_results('an industrial complex with large rectangular buildings', k=8)
show_results('a green meadow with no trees', k=8)


### Try your own query

Edit the string below and re-run. Things worth trying:

- Composite descriptions: *"a small river next to forest"*, *"farmland near a highway"*.
- Visual attributes: *"bright blue water"*, *"a dark green vegetation patch"*.
- Things EuroSAT shouldn't really contain: *"snow-covered mountain"*, *"a beach"* — what does the model surface as *closest*?


In [ ]:
show_results('river crossing farm land', k=10)
show_results('river crossing industrial complex', k=10)
show_results('river crossing residential area', k=10)

## 5. Quick quantitative check — does the ranking respect class labels?

For each class we form a simple prompt `"a satellite image of {class}"`, retrieve the top 50, and check what fraction actually have that label. This is a *very* loose retrieval metric — EuroSAT class names are not optimal CLIP prompts — but it gives a rough handle on quality.


In [ ]:
prompts = {
    'AnnualCrop':           'a satellite image of an annual crop field',
    'Forest':               'a satellite image of a dense forest',
    'HerbaceousVegetation': 'a satellite image of low herbaceous vegetation',
    'Highway':              'a satellite image of a highway',
    'Industrial':           'a satellite image of an industrial area',
    'Pasture':              'a satellite image of pasture land',
    'PermanentCrop':        'a satellite image of a permanent crop orchard',
    'Residential':          'a satellite image of a residential neighborhood',
    'River':                'a satellite image of a river',
    'SeaLake':              'a satellite image of a lake or sea',
}
K = 50
print(f'{"class":<22} P@{K}')
for cname, prompt in prompts.items():
    qvec = encode_text(prompt)
    top = np.argsort(-(img_embeddings @ qvec))[:K]
    cls_idx = class_names.index(cname)
    precision = (labels[top] == cls_idx).mean()
    bar = '█' * int(round(precision * 20))
    print(f'{cname:<22} {precision:>5.1%}  {bar}')


## 6. Takeaways

- **Text-to-image search collapses to a single mat-vec** once embeddings exist on both sides — milliseconds per query, at any scale that fits in RAM. This is the operational appeal of CLIP-style retrieval for EO.
- **Pre-computation is the trick.** Encoding the 5,400-image pool would have cost ~20 min on Colab CPU. We did it once with `scripts/build_eurosat_gitrsclip_embeddings.py`, pushed the 7.7 MB `.npz` to the HF Hub, and pulled it down via `hf_hub_download` — workshop run drops to <2 minutes.
- **Prompt sensitivity is real.** The P@50 table above will move several percentage points if you reword the prompts. In production, prompt engineering (or learned prompt templates) is part of the work.
- **Domain matters.** Git-RSCLIP is trained on remote-sensing image-text pairs; off-the-shelf OpenAI CLIP would be much worse on these queries. The pretraining corpus is the lever.
- **Open vocabulary, blurry boundaries.** The model will happily return *something* for queries that don't exist in the data ("a beach", "a snow-covered mountain"). That's a feature — useful for analog discovery — and a hazard — easy to mistake "closest in this dataset" for "actually matches the description".
- **Next:** Notebook 4 swaps the text encoder for the *image* encoder: same pool, same similarity formula, but the query is a satellite image instead of a string.
